In [1]:
### First read both revcomp and forward var effects
import pandas as pd

forward = pd.read_csv("/scratch/st-cdeboer-1/sambina/position_mpra/outputs/opentargets_model/gosai/k562_gosai.csv.gz", compression="gzip")
reverse = pd.read_csv("/scratch/st-cdeboer-1/sambina/position_mpra/outputs/opentargets_model/gosai/k562_gosai_revcomp.csv.gz", compression="gzip")

### Have to switch the column labels for the forward df
rename_dict = {}
for col in forward.columns:
    if col.startswith("offset_"):
        pos = int(col.split("_")[1])
        flipped_col = f"offset_{-pos}"
        rename_dict[col] = flipped_col

# Apply the renaming
forward = forward.rename(columns=rename_dict)

In [2]:
### Now change the values
offset_cols = [col for col in forward.columns if col.startswith("offset_")]
forward[offset_cols] = forward[offset_cols].applymap(lambda x: float(x.strip("[]")) if isinstance(x, str) else x)
forward["seq_id"] = ["seq_" + str(i + 1) for i in range(len(forward))]

reverse[offset_cols] = reverse[offset_cols].applymap(lambda x: float(x.strip("[]")) if isinstance(x, str) else x)
reverse["seq_id"] = ["seq_rev_" + str(i + 1) for i in range(len(reverse))]

/tmp/ipykernel_1883292/2042737668.py:3: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  forward[offset_cols] = forward[offset_cols].applymap(lambda x: float(x.strip("[]")) if isinstance(x, str) else x)
/tmp/ipykernel_1883292/2042737668.py:6: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  reverse[offset_cols] = reverse[offset_cols].applymap(lambda x: float(x.strip("[]")) if isinstance(x, str) else x)


In [3]:
### Combine and filter out the rows
forward["strand"] = "forward"
reverse["strand"] = "reverse"
combined = pd.concat([forward, reverse], ignore_index=True)
offset_cols = [col for col in combined.columns if col.startswith("offset_")]
# filtered = combined[combined[offset_cols].apply(
#     lambda row: (row > 0.15).any() or (row < -0.15).any(), axis=1)]
filtered = combined
print(len(filtered))

206706


In [4]:
### Now run PCA

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns


offset_cols = [col for col in filtered.columns if col.startswith("offset_")]
X = filtered[offset_cols].values

X_std = StandardScaler().fit_transform(X)

pca = PCA() 
X_pca = pca.fit_transform(X_std)

for i in range(X_pca.shape[1]):
    filtered[f'PC{i+1}'] = X_pca[:, i]

plt.figure(figsize=(7, 6))
sns.scatterplot(data=filtered, x="PC1", y="PC2", hue="strand", palette="Set2", alpha=0.8)
plt.title("PCA of Variant Effects by Strand Direction (Forward vs Reverse)")
plt.tight_layout()
plt.savefig('/scratch/st-cdeboer-1/sambina/position_mpra/outputs/cluster/PCA_full.svg', format='svg')
plt.show()

print("Explained variance (first few PCs):", pca.explained_variance_ratio_[:5])


TypeError: C function scipy.spatial._qhull._barycentric_coordinates has wrong signature (expected void (int, double *, double *, double *), got void (int, double *, double const *, double *))

### Color the dataframes by var effects

In [ ]:
# Identify offset columns again (just to be safe)
offset_cols = [col for col in combined.columns if col.startswith("offset_")]

# Add threshold column: check if *all* variant effects are within [-0.25, 0.25]
combined["var_threshold"] = combined[offset_cols].apply(
    lambda row: "Low effect (within ±0.25)" if row.abs().le(0.25).any() else "High effect (outside ±0.25)",
    axis=1
)


In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

X = combined[offset_cols].values
X_std = StandardScaler().fit_transform(X)

pca = PCA()
X_pca = pca.fit_transform(X_std)

for i in range(X_pca.shape[1]):
    combined[f"PC{i+1}"] = X_pca[:, i]

import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=combined,
    x="PC1",
    y="PC2",
    hue="var_threshold",
    palette={"Low effect (within ±0.25)": "#66c2a5", "High effect (outside ±0.25)": "#fc8d62"},
    alpha=0.8
)
plt.title("PCA of Variant Effects Colored by Effect Threshold")
plt.tight_layout()
plt.savefig("/scratch/st-cdeboer-1/sambina/position_mpra/outputs/cluster/PCA_var_effect_threshold.svg", format="svg")
plt.show()

print("Explained variance (first few PCs):", pca.explained_variance_ratio_[:5])


TypeError: C function scipy.spatial._qhull._barycentric_coordinates has wrong signature (expected void (int, double *, double *, double *), got void (int, double *, double const *, double *))

### UMAP clustering: try a different way of doing clustering

In [ ]:
import umap.umap_ as umap

# Initialize UMAP
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, metric='euclidean', random_state=42)

# Run UMAP on standardized data
X_umap = reducer.fit_transform(X_std)

# Add to your `combined` DataFrame
combined["UMAP1"] = X_umap[:, 0]
combined["UMAP2"] = X_umap[:, 1]

plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=combined,
    x="UMAP1",
    y="UMAP2",
    hue="var_threshold",
    palette={"Low effect (within ±0.25)": "#66c2a5", "High effect (outside ±0.25)": "#fc8d62"},
    alpha=0.4
)
plt.title("UMAP of Variant Effects Colored by Effect Threshold")
plt.tight_layout()
plt.savefig("/scratch/st-cdeboer-1/sambina/position_mpra/outputs/cluster/UMAP_var_effect_threshold.svg", format="svg")
plt.show()


TypeError: C function scipy.spatial._qhull._barycentric_coordinates has wrong signature (expected void (int, double *, double *, double *), got void (int, double *, double const *, double *))